In [4]:
import os
from circuit_tracer.subgraph.prune import prune_graph_pipeline, PruneGraph
from circuit_tracer.subgraph.api import generate_graph, get_feature, save_subgraph
from circuit_tracer.subgraph.classify import classify_features
from circuit_tracer.subgraph.group_llm import grouping_pipeline
from pathlib import Path
import torch

from circuit_tracer import ReplacementModel, attribute
from circuit_tracer.utils.create_graph_files import create_graph_files  
from circuit_tracer.graph import Graph, prune_graph, compute_graph_scores
from transformers import AutoModelForCausalLM, AutoTokenizer
import shap

## Load LLM

In [4]:
model_name = "google/gemma-2-2b" # google/gemma-scope-2-4b-it crosscoder/layer_9_17_22_29_width_65k_l0_medium
# transcoder_name = "mntss/clt-gemma-2-2b-2.5M"
# backend = 'transformerlens'  # change to 'nnsight' for the nnsight backend!
# model = ReplacementModel.from_pretrained(
#     model_name, transcoder_name, dtype=torch.bfloat16, backend=backend
# )

In [5]:
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(model_name).cuda()
# set model decoder to true
model.config.is_decoder = True
# ensure task_specific_params is initialized (avoid NoneType assignment error)
if model.config.task_specific_params is None:
    model.config.task_specific_params = {}
# set text-generation params under task_specific_params
model.config.task_specific_params["text-generation"] = {
    "do_sample": False, # set to False for deterministic output
    "max_new_tokens": 1, # set to 1 for single-token generation
    "temperature": 0, # set to 0 for deterministic output
    # "no_repeat_ngram_siz e": 2, 
}

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

## Use SHAP for token weights

In [6]:
explainer = shap.Explainer(model, tokenizer)

In [2]:
prompts = ['Fact: The capital of the state containing Dallas is']

In [8]:
shap_values = explainer(prompts)

In [9]:
print(shap_values)

.values =
array([[[-3.50681455e-06],
        [ 2.76400235e+00],
        [-8.72354661e-01],
        [ 2.00342429e+00],
        [ 4.72185472e+00],
        [ 1.18334559e+00],
        [ 1.11659276e-01],
        [ 2.30677287e+00],
        [ 9.67523081e-01],
        [ 5.83311160e+00],
        [ 2.01544604e+00]]])

.base_values =
array([[-21.22565525]])

.data =
(array(['', 'Fact', ':', ' The', ' capital', ' of', ' the', ' state',
       ' containing', ' Dallas', ' is'], dtype=object),)


In [ ]:
from scipy.special import softmax
token_weights = softmax(shap_values.values.squeeze()) # How to normalize Shap values? softmax, relu first then normalize, ...
print(token_weights)

[0.00198786 0.03153391 0.00083086 0.01473883 0.22338926 0.00649094
 0.00222269 0.01996207 0.0052309  0.67869559 0.01491708]


## Generate graph if not exist

In [ ]:
from generate_new_graphs import download_graph
download_graph(
    prompt=prompts[0],
    slug="austin",
    out_path="temp_graph_files/austin.json",
)

## ASG Pipeline

### Prune

In [3]:
json_path = "temp_graph_files/austin_plt.json"
# json_path = "temp_graph_files/future-qwen.json"
source_set = 'gemmascope-transcoder-16k' #'clt-hp' # gemmascope-transcoder-16k
# token_weights = [0.00198786, 0.03153391, 0.00083086, 0.01473883, 0.22338926, 0.00649094,
#  0.00222269, 0.01996207, 0.0052309, 0.67869559, 0.01491708]
token_weights = [0, 0, 0, 0, 1/3, 0, 0, 1/3, 0, 1/3, 0]
# token_weights = [0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,1/3,0,0,1/3,0,1/3,0,0,0,0,0,0,0,0,0,0,0]
PruneGraph = prune_graph_pipeline(
    json_path=json_path,
    logit_weights='target',
    token_weights=token_weights,
    node_influence_threshold=0.7,
    edge_influence_threshold=0.9,
    node_relevance_threshold=0.7,
    edge_relevance_threshold=0.9,
    keep_all_tokens_and_logits=False,
)

In [26]:
print(len(kept_ids))

65


In [27]:
for i, node_id in enumerate(kept_ids):
    print(node_id, attr[node_id].get("clerp", ""), node_inf[i].item(), node_rel[i].item())

16_25_9 Texas legal documents 0.503750205039978 0.6424384117126465
16_12678_9 cities 0.6242740750312805 0.6671419143676758
16_4298_10 capital 0.602944016456604 0.6816250681877136
16_13497_10 Numbers and parameters 0.6834822297096252 0.6882153153419495
17_7178_10 government buildings 0.5911982655525208 0.6703700423240662
18_1026_10 country names 0.627946674823761 0.6638426780700684
18_1437_10 Legal documents from Texas 0.5562974810600281 0.5726343393325806
18_3852_10 Locations 0.690534770488739 0.6541936993598938
18_5495_10 locations 0.5586398839950562 0.6529268622398376
18_6101_10 capital cities 0.5406162738800049 0.6111692190170288
18_8959_10 government/state 0.5327083468437195 0.583856463432312
18_16041_10 capital 0.6959729790687561 0.5978565812110901
19_7477_9 Dallas 0.6141869425773621 0.4621724486351013
19_37_10 Places 0.6009480953216553 0.6603857278823853
19_1445_10 Downtowns of cities 0.5513288974761963 0.5909120440483093
19_2439_10 Politics and government 0.6105824112892151 0.68

In [ ]:
from pathlib import Path
import torch

def save_prune_snapshot(
    out_path: str,
    PruneGraph,
):
    Path(out_path).parent.mkdir(parents=True, exist_ok=True)
    payload = {
        "kept_ids": PruneGraph.kept_ids,
        "pruned_adj": PruneGraph.pruned_adj.detach().cpu() if hasattr(PruneGraph.pruned_adj, "detach") else PruneGraph.pruned_adj,
        "node_inf": PruneGraph.node_influence.detach().cpu() if hasattr(PruneGraph.node_influence, "detach") else PruneGraph.node_influence,
        "node_rel": PruneGraph.node_relevance.detach().cpu() if hasattr(PruneGraph.node_relevance, "detach") else PruneGraph.node_relevance,
        "attr": PruneGraph.attr,
        "metadata": PruneGraph.metadata,
    }
    torch.save(payload, out_path)

def load_prune_snapshot(path: str):
    return torch.load(path, map_location="cpu")

In [28]:
save_prune_snapshot('subgraph/austin_plt.pt', kept_ids, pruned_adj, node_inf, node_rel, attr, metadata)

In [29]:
load_prune_snapshot('subgraph/austin_plt.pt')

{'kept_ids': ['16_25_9',
  '16_12678_9',
  '16_4298_10',
  '16_13497_10',
  '17_7178_10',
  '18_1026_10',
  '18_1437_10',
  '18_3852_10',
  '18_5495_10',
  '18_6101_10',
  '18_8959_10',
  '18_16041_10',
  '19_7477_9',
  '19_37_10',
  '19_1445_10',
  '19_2439_10',
  '19_2695_10',
  '19_7477_10',
  '20_15589_9',
  '20_114_10',
  '20_5916_10',
  '20_6026_10',
  '20_7507_10',
  '20_15276_10',
  '20_15589_10',
  '21_5943_10',
  '21_6316_10',
  '21_6795_10',
  '21_14975_10',
  '22_31_10',
  '22_3064_10',
  '22_3551_10',
  '22_4999_10',
  '22_11718_10',
  '23_2288_10',
  '23_6617_10',
  '23_11444_10',
  '23_12237_10',
  '23_12918_10',
  '23_13193_10',
  '23_13541_10',
  '23_13841_10',
  '23_15366_10',
  '24_709_10',
  '24_6044_10',
  '24_6394_10',
  '24_13277_10',
  '24_15013_10',
  '24_15627_10',
  '24_15694_10',
  '24_16258_10',
  '25_553_10',
  '25_583_10',
  '25_762_10',
  '25_2687_10',
  '25_4259_10',
  '25_4679_10',
  '25_4717_10',
  '25_4886_10',
  '25_13300_10',
  '25_16302_10',
  'E_

### Classify features

Heuristic classify alg is terrible, needs improvement or just use LLM

In [ ]:
# feature_types = classify_features(kept_ids, attr, metadata)
# print(feature_types)

IndexError: list index out of range

In [11]:
node_ids = [node_id for node_id in kept_ids if attr[node_id].get("feature_type", "") == "cross layer transcoder"]
# remove 21_61721_10, 22_74186_10
node_ids = [node_id for node_id in node_ids if node_id not in ["21_61721_10", "22_74186_10"]]
print(node_ids)

['16_89970_9', '18_45367_10', '18_56027_10', '19_8271_10', '19_62362_10', '20_44686_9', '20_14775_10', '20_44686_10', '20_74108_10', '21_16875_10', '21_84338_10', '22_11998_10', '22_26199_10', '22_32893_10', '22_43081_10', '22_81571_10', '23_29602_10', '23_31673_10', '23_59746_10', '24_79427_10']


In [8]:
labels = {node_id: attr[node_id].get("clerp", "") for node_id in node_ids }
print(labels)

{'16_89970_9': 'Texas', '18_45367_10': 'US cities', '18_56027_10': 'Dallas', '19_8271_10': 'Foreign words', '19_62362_10': 'city names', '20_44686_9': 'Texas locations/legal contexts', '20_14775_10': 'XML code', '20_44686_10': 'Texas locations/legal contexts', '20_74108_10': 'capital', '21_16875_10': 'cities', '21_84338_10': 'Cities and locations', '22_11998_10': 'Texas and Dallas', '22_26199_10': 'Texas', '22_32893_10': 'Texas legal documents', '22_43081_10': 'technical contexts', '22_81571_10': 'Texas locations and organizations', '23_29602_10': 'City or region names', '23_31673_10': 'Court appeals at specific location', '23_59746_10': 'Technical/Scientific writing', '24_79427_10': ' Kara'}


In [9]:
feature_types = {
    node_id: "Abstract" for node_id in node_ids
}
feature_types['1_89326_9'] = "Input"
print(feature_types)

{'16_89970_9': 'Abstract', '18_45367_10': 'Abstract', '18_56027_10': 'Abstract', '19_8271_10': 'Abstract', '19_62362_10': 'Abstract', '20_44686_9': 'Abstract', '20_14775_10': 'Abstract', '20_44686_10': 'Abstract', '20_74108_10': 'Abstract', '21_16875_10': 'Abstract', '21_84338_10': 'Abstract', '22_11998_10': 'Abstract', '22_26199_10': 'Abstract', '22_32893_10': 'Abstract', '22_43081_10': 'Abstract', '22_81571_10': 'Abstract', '23_29602_10': 'Abstract', '23_31673_10': 'Abstract', '23_59746_10': 'Abstract', '24_79427_10': 'Abstract', '1_89326_9': 'Input'}


In [12]:
supernodes = grouping_pipeline(
    node_ids = node_ids,
    labels = labels,
    feature_types = feature_types,
    prompt = prompts[0],
    model_name = 'gemini-2.5-flash',
)
print(supernodes)

[['Texas State Context', '16_89970_9', '22_11998_10', '22_26199_10'], ['General Geographic Entities', '18_45367_10', '19_62362_10', '21_16875_10', '21_84338_10', '23_29602_10'], ['Texas Legal/Specific Contexts', '20_44686_9', '20_44686_10', '22_32893_10', '22_81571_10', '23_31673_10'], ['Miscellaneous/Irrelevant', '19_8271_10', '20_14775_10', '22_43081_10', '23_59746_10', '24_79427_10']]


### Visualize on the web

In [ ]:
import json
from circuit_tracer.subgraph.api import save_subgraph

# Expected format for save_subgraph:
# supernodes = [
#   ["label_1", "node_id_a", "node_id_b"],
#   ["label_2", "node_id_c", "node_id_d", "node_id_e"],
# ]

status, body = save_subgraph(
    modelId="gemma-2-2b",
    slug="austin",                       # parent graph slug
    displayName="austin-grouped-v1",     # subgraph name shown on Neuronpedia
    pinnedIds=PruneGraph.kept_ids,                  # or PruneGraph.kept_ids
    supernodes=supernodes,               # output from grouping pipeline
    pruningThreshold=0.7,
    densityThreshold=0.99,
)

print("status:", status)
try:
    print(json.loads(body))
except Exception:
    print(body)

# TODO
List of things that need more experiments/improvements in descending order of importance for each step
## Pruning
1. Choose threshold (currently just sweep the threshold until we have ~30 nodes)
2. Initialize with Shap (normalize) (softmax | relu + normalize | +base_value then normalize)
3. Normalize adjacency matrix (currently adjacency.abs() then normalize (negative edges still contribute to influence and relevance))

## Grouping

Objectives to group features:
- similar embeddings (in classic SAE usually decoder vectors)
- simliar edges/logit effects (promoting the same feature or the same logit)
- similar contexts (activating on similar tokens, concepts, contexts)
- based on the claim we make about the mechanism (we might want to preserve the paths)

Approaches tried:
- Greedy constraint grouping
    + built distance graph based on BERT embeddings of autointerp (+ feature type) and group with antichain constraint (don't group nodes with path to each other)
    + input: autointerp + adj matrix
    + problems: constraints too strict, sensitive on the edges of the graph; did not consider local role of the features.
- Prompt LLM: 
    + Classify feature type based on feature details
    + Group using feature types + autointerp
    + problems: cover context and global role of the features, but not local role (in this exact graph) -> might not be mechanisticly sensible
    
Approaches to try:
- Clustering by (decoder vectors or autointerp) + edge weights similarity, constraint by difference in layers (may be sweep diff_layer = 0, 1, ... and merge iteratively)
- Higher order graphs




## Eval
1. Test intervention api
2. Find dataset
3. Hypothesis testing: Mechanism Preservation, Mechanism Localization, Minimality